# 低净码率局部时间标签 V1 — 固定开发验证
6bit ID1..21分别表示8帧名义单元，32chip RM(1,5)/r7 +独立16chip周期pilot。normalized LLL ch4..15/g1..21，alpha=.15，balanced orthogonal Walsh PN，新方法补定，不是旧论文结果修复。原净32位不重写。只写一次delta，不加幅、补偿、扫描或训练；native保护user ch0..3及边界，实际VAE耦合仍须检验。
局部52真实pixel帧→固定8相位各E45(T12)，读Haar j1..4。pilot全部64signed correlation均值唯一最大排序；该候选还须>=3行all16sign通过，RM不救phase。码字正确不是检测，>=2一致行的origin仅为CONDITIONAL_UNCALIBRATED临时提议；保留并列、未知、混合、缺失与错误合法ID。
4已见开发源，不是独立确认；私有crop origin48,49,50,51由制备器使用，receiver不收真origin/ID。所有raw、blind、payload保存/回读后才postjoin已制备几何与message。真实局部E不由full-latent切窗替代。
P0/TAG同runtime D8/quant8/MP4各8，fullE8，OFF历史actual MP4仅背景；localE45=96，E总104。FFT72/keyed payload80/Counter11776；1608raw+24family/96final slots。localTAG/correctkey128行仅16nominalFULL、112MIXED，不能事后删混合。原32退步、错钥背景、missing-ID误接受、全图/时序质量另报，无PASS/FPR或完成时间路径的主张。
Notebook直接从RICHAAARC/SC-SSTW现有dev/local-joint-state-payload-v1分支加载可用源码，Run all执行固定任务，无需手动上传ZIP，无摘要、版本或Git-clean门禁。GPU由用户运行；代理只做CPU/静态/stub。
一次attempt，STARTED先持久化，失败/中断消费；后续Run all仅保存JSON只读，不重试物理。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Lowrate-Local-Time-Label-V1/fixed-viewed-development-v1')
def execution_artifacts(output):
    out=Path(output);found=[]
    for name in ('attempt.json','ledger.json','result.json','evaluation.json','plan.json','fixed_rows.json','progress.json','post_truth_access.json','message_post_rows.json','payload_group_pairs.json','payload_cell_pairs.json'):
        for suffix in ('','.tmp'):
            path=out/(name+suffix)
            if path.exists() or path.is_symlink():found.append(name+suffix)
    for name in ('raw','blind','payload','private','PN','construction','native','latents','media','quality','markers','receipts'):
        root=out/name
        if root.exists():found.extend(str(x.relative_to(out)) for x in root.rglob('*') if x.is_file() or x.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT) or (OUTPUT/'families').exists()
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-lowrate-local-time-label-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch','dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_lowrate_local_time_label_v1_run.py').is_file():
        raise FileNotFoundError('Available source lacks low-rate entry; publish authorized source or use editable SOURCE_ROOT: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
else:
    print('Existing scientific attempt: skip source/setup/model/physical; saved JSON read-only.')


固定执行。仅修复真实缺失依赖和导入故障，不绑定精确版本或GPU型号。

In [ ]:

import subprocess, sys, time
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','update'], 'refresh package metadata for missing FFmpeg')
        command(['apt-get','install','-y','ffmpeg'], 'install missing ffmpeg/ffprobe')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable after dependency installation; see '+str(LOG))
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE dependencies')
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_lowrate_local_time_label_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress.get('counts',{}).get('RETURNED',0), '/', progress['planned'], 'counts', progress.get('counts',{}), 'active', progress.get('active'), flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        print('Stopped; saved attempts consumed. Later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


保存结果只读，缺失与失败仍属于固定分母。

In [ ]:

for name in ('result.json','evaluation.json'):
    path=OUTPUT/name
    if not path.is_file():print('Unavailable',name);continue
    try:value=json.loads(path.read_text())
    except (OSError,ValueError) as e:print('Saved read error',name,str(e));continue
    if name=='result.json':print('Execution',value.get('status'),'physical calls',value.get('kinds_started'),'fixed',value.get('fixed_denominators'))
    else:
        print('Tag',value.get('tag_summary'));print('Local final',value.get('local_final'));print('Families',value.get('families'));print('Original32',value.get('payload'));print('Truth issues',value.get('issues'))
print('Costs: construction/. Quality/temporal/clipping: quality/. All failure receipts retained. Output:',OUTPUT)
print('Conditional provisional origin is not authentication or completed time path; four sources are seen development.')
